# Comprehensive Statistical Analysis and Pilot Modelling of MDR-TB Treatment Outcomes
## (CSC 6701: Descriptive Statistics, Inferential Statistics & Machine Learning)

**Objective:**
The aim of this analysis is to:
- Understand the structure of the MDR-TB dataset.
- Summarize main patient characteristics (Descriptive Statistics).
- Describe outcome patterns and identify potential predictors of mortality.
- Check variation, skewness, missingness, and outliers.
- Investigate relationships between clinical variables and treatment success.
- Perform inferential statistics to determine if observed differences are statistically meaningful.
- Train a pilot Random Forest model and visualize feature importance.

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind, f_oneway
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Set display options for readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_theme(style="whitegrid")

## 3. Load the Dataset
This notebook is designed to work in Google Colab. You can upload your own file or load the reconstructed mock dataset.

In [ ]:
import os

# Check if running in Google Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

filename = "drtb_central_zambia_reconstructed_mock.csv"

if IN_COLAB and not os.path.exists(filename):
    print("Please upload the dataset file.")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]

try:
    if filename.endswith('.csv'):
        df = pd.read_csv(filename)
    else:
        df = pd.read_excel(filename)
    print(f"Successfully loaded {filename}")
except Exception as e:
    print(f"Error loading file: {e}")

# Display basic info
print(f"\nDataset Shape: {df.shape}")
display(df.head())
display(df.info())

## 4. Data Understanding
**Observational Unit:** One row represents one patient record.

**Statistical Definitions:**
- **Population:** Patients with Drug-Resistant Tuberculosis (DR-TB) in Central Province, Zambia.
- **Sample:** The 183 records available in this reconstructed dataset.
- **Parameter:** The true, unknown population mortality rate or mean age.
- **Statistic:** The values calculated from this sample (e.g., sample mean age, sample mortality proportion).

In [ ]:
# Classifying Variables
categorical_nominal = ['gender', 'district', 'hiv_status', 'drtb_type', 'registration_group', 'outcome']
categorical_ordinal = ['age_group'] # Age groups are ordered bands
numerical_discrete = ['year_of_diagnosis']
numerical_continuous = ['age_years']

# Data Quality Summary Table
quality_summary = pd.DataFrame({
    'Column Name': df.columns,
    'Data Type': df.dtypes,
    'Unique Values': [df[col].nunique() for col in df.columns],
    'Missing Values': [df[col].isnull().sum() for col in df.columns],
    'Percentage Missing': [f"{(df[col].isnull().sum() / len(df) * 100):.1f}%" for col in df.columns]
})
display(quality_summary)

## 5. Data Cleaning and Quality Checks

In [ ]:
# 1. Check for duplicates
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

# 2. Standardize categorical columns
for col in categorical_nominal:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()
        
# 3. Handle missing values
missing_counts = df.isnull().sum()
if missing_counts.sum() == 0:
    print("No missing values detected.")
else:
    print("Missing values summary:")
    print(missing_counts[missing_counts > 0])

## 6. Descriptive Statistics
### 6.1 Categorical Summaries

In [ ]:
def summarize_categorical(column):
    counts = df[column].value_counts()
    pcts = df[column].value_counts(normalize=True) * 100
    summary = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts.round(1)})
    
    print(f"\nSummary for {column}:")
    display(summary)
    
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x=column, palette="viridis")
    plt.title(f"Distribution of {column.replace('_', ' ').title()}")
    plt.xticks(rotation=45)
    plt.show()

for col in ['gender', 'hiv_status', 'outcome', 'registration_group']:
    summarize_categorical(col)

### 6.2 Numerical Summaries

In [ ]:
def summarize_numerical(column):
    desc = df[column].describe()
    skew = df[column].skew()
    kurt = df[column].kurtosis()
    
    print(f"\nNumerical Summary for {column}:")
    print(desc)
    print(f"Skewness: {skew:.3f}")
    print(f"Kurtosis: {kurt:.3f}")
    
    if abs(skew) < 0.5:
        print("Interpretation: The distribution is roughly symmetric. Mean is a good measure of central tendency.")
    else:
        print("Interpretation: The distribution is skewed. Median may be a more robust measure of central tendency.")

summarize_numerical('age_years')

## 7. Visual Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.histplot(df['age_years'], kde=True, color="skyblue")
plt.title("Histogram of Age")

plt.subplot(1, 2, 2)
sns.boxplot(y=df['age_years'], color="lightcoral")
plt.title("Boxplot of Age")

plt.tight_layout()
plt.show()

## 8. Outliers and Z-Scores

In [ ]:
# IQR Method
Q1 = df['age_years'].quantile(0.25)
Q3 = df['age_years'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = df[(df['age_years'] < lower_bound) | (df['age_years'] > upper_bound)]
print(f"Outliers detected via IQR: {len(outliers_iqr)}")

# Z-Score Method
df['age_zscore'] = (df['age_years'] - df['age_years'].mean()) / df['age_years'].std()
outliers_z = df[abs(df['age_zscore']) > 3]
print(f"Outliers detected via Z-score (>3 std): {len(outliers_z)}")

## 9. Outcome Analysis and Binary Variable

In [ ]:
# Create Binary Poor Outcome Variable
poor_outcome_labels = ['Died', 'Lost to Follow Up']
df['poor_outcome'] = df['outcome'].apply(lambda x: 1 if x in poor_outcome_labels else 0)

print("Binary Poor Outcome Distribution:")
print(df['poor_outcome'].value_counts(normalize=True).round(3) * 100)

# Outcome by Gender
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='gender', hue='outcome')
plt.title("Treatment Outcome by Gender")
plt.legend(title="Outcome", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

## 10. Confidence Intervals

In [ ]:
def confidence_interval_proportion(p, n, confidence=0.95):
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    se = np.sqrt((p * (1 - p)) / n)
    return p - z * se, p + z * se

p_poor = df['poor_outcome'].mean()
n = len(df)
low, high = confidence_interval_proportion(p_poor, n)
print(f"95% Confidence Interval for Proportion of Poor Outcomes: [{low:.3f}, {high:.3f}]")
print(f"Interpretation: We are 95% confident that the true population proportion lies between {low:.1%} and {high:.1%}.")

## 11. Hypothesis Testing
### Test: HIV Status and Poor Outcome
**H0:** Poor outcome is independent of HIV status.  
**H1:** Poor outcome is associated with HIV status.

In [ ]:
contingency = pd.crosstab(df['hiv_status'], df['poor_outcome'])
chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2:.3f}")
print(f"P-value: {p:.4f}")

if p < 0.05:
    print("Decision: Reject H0. Association exists.")
else:
    print("Decision: Fail to reject H0. No association found.")

## 12. Preliminary Model Training (Random Forest)
Following the statistical analysis, we build a pilot Random Forest model to predict poor outcomes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# Preprocessing
features = ['age_years', 'is_male', 'hiv_positive', 'registration_group', 'drtb_type']
X = pd.get_dummies(df[features], drop_first=True)
y = df['poor_outcome']

# Exclude 'Still on Treatment' for training if necessary
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest Performance:")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.3f}")

# Feature Importance
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=importances, y=importances.index, palette="magma")
plt.title("Feature Importance (Pilot Random Forest)")
plt.xlabel("Gini Importance")
plt.show()

## 13. Final Summary and Model Readiness
**Summary of Findings:**
1. **Sample Size:** 183 patients recorded from 2017 to 2021.
2. **Demographics:** Mean age is approximately 35.2 years; Male patients predominate (57.9%).
3. **Outcome Distribution:** Mortality rate is significant (~21%).

**Model Readiness Check:**
- **Class Imbalance:** Poor outcomes represent ~27% of the data.
- **Verdict:** The dataset is ready for pilot predictive modeling using tree-based algorithms like Random Forest.